# Analyse métier - template Polars

Notebook d'exploration : lecture CSV, colonnes dérivées, contrôle qualité, graphique Altair.

In [1]:
from pathlib import Path
import sys

import altair as alt
import polars as pl

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from libs.cleaners import normalize_amount, normalize_date

🎉 libs.cleaners


In [2]:
input_path = Path("input/commandes_input_exemple.csv")
df = pl.read_csv(input_path)
df

commande_id,client,email,montant,date_commande,statut
i64,str,str,str,str,str
1001,"""Acme""","""orders@acme.com""","""1 200,50 €""","""21/03/2021""","""ok"""
1002,"""Beta Corp""","""contact_beta.fr""","""980.00""","""2026-03-02""","""ok"""
1003,"""Gamma SA""","""gamma@gamma.com""",null,"""2026-03-03""","""pending"""
1004,"""Delta""","""delta@delta.com""","""450,00""","""2026-03-04""","""ok"""


In [3]:
df.schema, df.null_count()

(Schema([('commande_id', Int64),
         ('client', String),
         ('email', String),
         ('montant', String),
         ('date_commande', String),
         ('statut', String)]),
 shape: (1, 6)
 ┌─────────────┬────────┬───────┬─────────┬───────────────┬────────┐
 │ commande_id ┆ client ┆ email ┆ montant ┆ date_commande ┆ statut │
 │ ---         ┆ ---    ┆ ---   ┆ ---     ┆ ---           ┆ ---    │
 │ u64         ┆ u64    ┆ u64   ┆ u64     ┆ u64           ┆ u64    │
 ╞═════════════╪════════╪═══════╪═════════╪═══════════════╪════════╡
 │ 0           ┆ 0      ┆ 0     ┆ 1       ┆ 0             ┆ 0      │
 └─────────────┴────────┴───────┴─────────┴───────────────┴────────┘)

In [4]:
df_clean = (
    df.with_columns([
        pl.col("email").str.strip_chars().str.to_lowercase().alias("email_clean"),
        pl.col("montant").map_elements(normalize_amount, return_dtype=pl.Float64).alias("montant_net"),
        pl.col("date_commande").map_elements(normalize_date, return_dtype=pl.Utf8).alias("date_commande_clean"),
        pl.col("statut").str.strip_chars().str.to_lowercase().alias("statut_normalise"),
    ])
    .with_columns([
        (
            pl.col("email_clean").str.contains("@", literal=True)
            & pl.col("email_clean").str.contains(".", literal=True)
        ).alias("email_valide"),
        (
            pl.col("montant_net").is_not_null()
            & pl.col("date_commande_clean").is_not_null()
        ).alias("ligne_valide"),
    ])
)
df_clean

commande_id,client,email,montant,date_commande,statut,email_clean,montant_net,date_commande_clean,statut_normalise,email_valide,ligne_valide
i64,str,str,str,str,str,str,f64,str,str,bool,bool
1001,"""Acme""","""orders@acme.com""","""1 200,50 €""","""21/03/2021""","""ok""","""orders@acme.com""",1200.5,"""2021-03-21""","""ok""",true,true
1002,"""Beta Corp""","""contact_beta.fr""","""980.00""","""2026-03-02""","""ok""","""contact_beta.fr""",980.0,"""2026-03-02""","""ok""",false,true
1003,"""Gamma SA""","""gamma@gamma.com""",null,"""2026-03-03""","""pending""","""gamma@gamma.com""",null,"""2026-03-03""","""pending""",true,false
1004,"""Delta""","""delta@delta.com""","""450,00""","""2026-03-04""","""ok""","""delta@delta.com""",450.0,"""2026-03-04""","""ok""",true,true


In [5]:
quality = (
    df_clean.group_by("statut_normalise")
    .agg([
        pl.len().alias("nb_lignes"),
        pl.col("ligne_valide").sum().alias("nb_valides"),
    ])
    .sort("nb_lignes", descending=True)
)
quality

statut_normalise,nb_lignes,nb_valides
str,u64,u64
"""ok""",3,3
"""pending""",1,0


In [13]:
chart =  (
    quality.plot.bar(
        x="statut_normalise",
        y="nb_lignes",
    )
    .properties(width=500, title="Irises")
    .configure_scale(zero=False)
    .configure_axisX(tickMinStep=1)
)
chart.encoding.x.title = "Statut"
chart.encoding.y.title = "Nombre de lignes"
chart

alt.Chart(...)

In [14]:
alt.Chart(quality).mark_bar().encode(
    x=alt.X("statut_normalise:N", title="Statut"),
    y=alt.Y("nb_lignes:Q", title="Nombre de lignes"),
    tooltip=["statut_normalise", "nb_lignes", "nb_valides"],
).properties(title="Répartition des statuts", width=500)

alt.Chart(...)